# Notebook 7 of 7: Limitations, Ethics, and Opportunities

## The Final Chapter in Our AI Journey

---

*In this final notebook, we step back from the code and ask the bigger questions: What can AI really do? Where does it fall short? And what does it mean for all of us?*

*This notebook is mostly reading and reflection, with a few code demonstrations to ground our discussion in the project we've built together.*

## 1. Introduction: The Bigger Picture

Over the past 6 notebooks, you've built AI models from scratch, watched them learn, seen them fail, and improved them. You now understand the fundamental mechanism behind ChatGPT, Claude, and every major AI system: **predict the next word, billions of times.**

That's it. That's the core idea. Whether it's a simple RNN stumbling through its first Italian words, an LSTM holding onto longer memories, or a massive Transformer attending to everything at once -- the underlying principle is the same:

> **Given what came before, what word is most likely to come next?**

You've seen how this simple idea, scaled up with better architectures and smarter sampling strategies, can produce text that feels surprisingly human.

**Now let's talk about what this means for the real world.**

Because understanding how AI works is only half the story. Understanding its **limitations**, its **ethical implications**, and its **genuine potential** -- that's what turns knowledge into wisdom.

In this final notebook, we'll be honest about five important limitations of AI, and then we'll explore the genuine opportunities it creates. No hype, no fear -- just a clear-eyed look at where we are and where we might be going.

---

## 2. Limitation 1: AI Doesn't Understand -- It Predicts

This is perhaps the most important thing to internalize:

**Our model learned to write Italian-sounding text, but it doesn't UNDERSTAND Italian.**

It doesn't know what love means. It doesn't know what a sunset looks like. It doesn't know why a song is beautiful or why a lyric makes someone cry. It has never experienced anything at all.

What it *did* learn are **statistical patterns**:
- After the word *"amore"* (love), the word *"mio"* (my) is very likely
- After *"il cielo"* (the sky), words like *"blu"* (blue) or *"stellato"* (starry) often follow
- Song lyrics tend to have short lines and repeat certain phrases

The model is essentially a very sophisticated pattern-matching machine. It learned which words tend to appear near which other words, and it uses that knowledge to generate new sequences.

Let's see this in action. We'll load our model architecture, generate some text, and examine what comes out.

In [ ]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer
import warnings
warnings.filterwarnings('ignore')

# Load the same tokenizer we've used throughout the series
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Define our LSTM model (same architecture from Notebook 4)
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        embeds = self.dropout(self.embedding(x))
        lstm_out, hidden = self.lstm(embeds, hidden)
        output = self.fc(self.dropout(lstm_out))
        return output, hidden

# Create the model
vocab_size = tokenizer.vocab_size
model = LSTMModel(vocab_size=vocab_size, embed_dim=256, hidden_dim=512)

# Generation function with top-k sampling (from Notebook 6)
def generate_text(model, tokenizer, prompt, max_length=50, temperature=0.8, top_k=40):
    """Generate text one word at a time -- exactly how our models work."""
    model.eval()
    tokens = tokenizer.encode(prompt, return_tensors='pt')
    hidden = None

    with torch.no_grad():
        for _ in range(max_length):
            output, hidden = model(tokens[:, -1:] if hidden else tokens, hidden)
            logits = output[:, -1, :] / temperature

            # Top-k filtering: only consider the 40 most likely next words
            top_k_logits, top_k_indices = torch.topk(logits, top_k)
            probs = torch.softmax(top_k_logits, dim=-1)
            next_idx = torch.multinomial(probs, 1)
            next_token = top_k_indices.gather(1, next_idx)

            tokens = torch.cat([tokens, next_token], dim=1)

    return tokenizer.decode(tokens[0], skip_special_tokens=True)

# Generate lyrics from different prompts
# Note: Without trained weights, the model produces random text.
# With trained weights, it produces Italian-sounding but often incoherent text.
# Either way, it perfectly illustrates our point!

prompts = ["Amore mio", "La notte", "Nel silenzio"]

print("=" * 65)
print("  GENERATED LYRICS -- Does the model 'understand' these words?")
print("=" * 65)

for prompt in prompts:
    generated = generate_text(model, tokenizer, prompt, max_length=30)
    print(f"\n  Prompt: '{prompt}'")
    print(f"  Output: {generated}")
    print(f"  {'~' * 60}")

print(f"\n  Notice: the model assembles words based on statistical")
print(f"  patterns, not meaning. It has no idea what these words")
print(f"  refer to in the real world.")
print("=" * 65)

### What to notice in the output above

Look at the generated text and ask yourself these questions:

- **Does it make logical sense?** Can you follow a coherent thought from beginning to end?
- **Does it convey a real emotion?** Or does it just string together emotional-sounding words?
- **Would a human songwriter write this?** Or is it pretty-sounding word salad?

Even with a well-trained model, you'd see text like:

> *"Amore mio, nel cielo della notte il mare canta le stelle..."*
> *(My love, in the sky of the night the sea sings the stars...)*

This sounds poetic! But the sea doesn't sing stars. There's no coherent thought connecting these images. The model has no **intent**, no **meaning**, no **experience** behind its words. It's assembling statistically likely word sequences, nothing more.

**This is the fundamental limitation of current AI.** When ChatGPT gives you a confident-sounding answer, it's doing the same thing our little model does -- just at a much larger scale. It predicts likely next words. It doesn't "know" if what it's saying is true, meaningful, or helpful. It just knows what words tend to follow other words in its training data.

There is active debate among researchers about whether scaling up this approach could eventually lead to something closer to "understanding," or whether a fundamentally different approach is needed. That question remains open.

---

## 3. Limitation 2: Bias In = Bias Out

Here's a simple truth about AI that has enormous consequences:

**An AI model can only reflect what's in its training data.**

If you train a model exclusively on Italian pop love songs from the 2000s, it will only be able to generate Italian pop love songs from the 2000s. It won't write rap, opera, folk music, or protest songs -- not because those genres are less valid, but because the model has never seen them.

This is exactly how bias works in real-world AI systems:
- A hiring AI trained mostly on resumes from male candidates may learn to favor male applicants
- A medical AI trained mostly on data from one ethnic group may perform poorly for others
- A language model trained mostly on English text will be far less capable in other languages

**The bias isn't malicious -- it's mathematical.** The model learns the patterns in the data. If the data is skewed, the model is skewed.

Let's look at our own dataset to understand what biases it might contain. What genres are represented? What themes dominate? What's missing?

In [ ]:
from collections import Counter
import os

# Load our lyrics dataset
data_path = os.path.join('..', 'data', 'italian_lyrics.txt')

try:
    with open(data_path, 'r', encoding='utf-8') as f:
        lyrics_lines = f.readlines()
    print("Loaded lyrics dataset successfully.\n")
except FileNotFoundError:
    # If the file isn't available, use sample data for demonstration
    print("Note: Lyrics file not found. Using sample data for demonstration.\n")
    lyrics_lines = [
        "Ti amo con tutto il cuore amore mio per sempre nella vita",
        "La notte stellata brilla sopra di noi nel cielo blu infinito",
        "Canto per te una canzone dolce come il miele della primavera",
        "Nel silenzio della sera il vento porta via i sogni miei",
        "Insieme camminiamo sotto la luna piena verso il mare",
    ] * 200

# ========================================
# BASIC DATASET STATISTICS
# ========================================
print("=" * 60)
print("  OUR DATASET AT A GLANCE")
print("=" * 60)

total_songs = len(lyrics_lines)
word_counts = [len(line.split()) for line in lyrics_lines]
total_words = sum(word_counts)
avg_words = total_words / total_songs if total_songs > 0 else 0

print(f"\n  Total songs:            {total_songs:,}")
print(f"  Total words:            {total_words:,}")
print(f"  Average words per song: {avg_words:.1f}")
print(f"  Shortest song:          {min(word_counts)} words")
print(f"  Longest song:           {max(word_counts)} words")

# ========================================
# WORD FREQUENCY ANALYSIS
# ========================================
print("\n" + "=" * 60)
print("  MOST COMMON WORDS -- What themes dominate?")
print("=" * 60)

# Combine all lyrics and count words
all_words = []
for line in lyrics_lines:
    words = line.lower().strip().split()
    all_words.extend(words)

word_freq = Counter(all_words)
top_30 = word_freq.most_common(30)

# Approximate translations for common Italian words
translations = {
    'che': 'that/what', 'non': 'not', 'il': 'the (m)', 'la': 'the (f)',
    'di': 'of', 'un': 'a/an', 'mi': 'me/my', 'per': 'for',
    'e': 'and', 'io': 'I', 'tu': 'you', 'si': 'yes/oneself',
    'ti': 'you (obj)', 'come': 'like/how', 'con': 'with', 'se': 'if',
    'ma': 'but', 'nel': 'in the', 'lo': 'it/the', 'le': 'the (f.pl)',
    'del': 'of the', 'da': 'from', 'ci': 'us/there', 'a': 'to/at',
    'sei': 'you are', 'al': 'to the', 'una': 'a (f)', 'solo': 'only/alone',
    'amore': 'love', 'cuore': 'heart', 'vita': 'life', 'notte': 'night',
    'mai': 'never', 'tutto': 'everything', 'questa': 'this (f)',
    'sono': 'I am', 'ho': 'I have', 'mio': 'my (m)', 'te': 'you',
    'pi\u00f9': 'more', 'cosa': 'thing/what', 'mondo': 'world',
    'occhi': 'eyes', 'tempo': 'time', 'sempre': 'always',
}

print(f"\n  {'Rank':<6} {'Word':<15} {'Count':<10} {'Meaning (approx.)'}")
print(f"  {'-' * 55}")

for i, (word, count) in enumerate(top_30, 1):
    meaning = translations.get(word, '')
    print(f"  {i:<6} {word:<15} {count:<10} {meaning}")

print(f"\n  Total unique words: {len(word_freq):,}")

# ========================================
# THEMATIC ANALYSIS
# ========================================
print("\n" + "=" * 60)
print("  THEMATIC WORD GROUPS -- What is our AI's 'worldview'?")
print("=" * 60)

# Count words in thematic groups
themes = {
    "Love & Romance": ['amore', 'cuore', 'bacio', 'amare', 'amato', 'amata',
                        'innamorato', 'passione', 'desiderio', 'baci'],
    "Nature & Sky":   ['cielo', 'mare', 'sole', 'luna', 'stelle', 'vento',
                        'terra', 'notte', 'acqua', 'pioggia'],
    "Emotion & Pain": ['dolore', 'lacrime', 'piangere', 'triste', 'paura',
                        'rabbia', 'felice', 'gioia', 'speranza', 'sogno'],
    "Time & Memory":  ['tempo', 'sempre', 'mai', 'ieri', 'domani', 'oggi',
                        'ricordo', 'memoria', 'momento', 'attimo'],
}

for theme_name, theme_words in themes.items():
    total = sum(word_freq.get(w, 0) for w in theme_words)
    top_in_theme = [(w, word_freq.get(w, 0)) for w in theme_words if word_freq.get(w, 0) > 0]
    top_in_theme.sort(key=lambda x: -x[1])
    top_3 = ', '.join(f'{w} ({c})' for w, c in top_in_theme[:3])
    print(f"\n  {theme_name}: {total} total mentions")
    if top_3:
        print(f"    Top words: {top_3}")

### What the data reveals about bias

Look at the word frequencies and thematic analysis above. You'll likely notice:

- **Love and romance dominate** -- words like *amore*, *cuore*, *vita* appear far more than words about politics, work, or social issues
- **Certain emotions are overrepresented** -- longing and sadness appear more than joy or anger
- **The vocabulary reflects specific genres** -- pop ballads and love songs, not rap or folk music

Now imagine the consequences of this at a much larger scale:

| If training data is mostly... | The AI will tend to... |
|-------------------------------|----------------------|
| Love songs | Generate romantic language, miss other emotions |
| Male songwriters | Reflect male perspectives more than female ones |
| Songs from 2000-2020 | Use modern Italian, miss historical styles |
| Pop and rock genres | Struggle with rap, folk, or classical styles |
| Northern Italian artists | Underrepresent Southern Italian dialect and culture |

**This is not hypothetical.** Real AI systems deployed today have exactly these kinds of blind spots, just in higher-stakes domains: healthcare, criminal justice, lending, and hiring.

The lesson: **always ask what's in the training data -- and what's missing.**

---

## 4. Limitation 3: Hallucination

You may have heard this term in the news: AI systems "hallucinate." It sounds dramatic, but it's actually a straightforward consequence of how these models work.

**Hallucination means the model generates text that looks real and sounds confident -- but is factually wrong, internally contradictory, or simply made up.**

Why does this happen? Go back to the core mechanism: the model predicts the most likely next word. It has no fact-checking system. It has no database of true statements. It has no way to verify whether what it's writing actually makes sense or corresponds to reality.

In our Italian lyrics project, hallucination looks like:
- Combining Italian words in ways that are **grammatically incorrect** (wrong gender agreement, impossible verb conjugations)
- Creating **nonsensical imagery** that no real songwriter would use
- Generating **made-up words** that look Italian but aren't real

In large language models like ChatGPT or Claude, hallucination looks like:
- Citing **academic papers that don't exist** (with realistic-sounding titles and authors)
- Providing **incorrect dates, statistics, or facts** with complete confidence
- Making up **quotes** that a person never actually said

Let's generate several examples and examine which parts seem plausible versus nonsensical.

In [ ]:
# Generate multiple examples to examine for hallucination
# We'll use different prompts and temperatures to show variety

print("=" * 65)
print("  HALLUCINATION EXAMPLES")
print("  Generated text that sounds real but may not make sense")
print("=" * 65)

# Different prompts that might trigger different kinds of output
test_prompts = [
    ("Nella città di", "A place-based prompt"),
    ("Io canto perché", "A reason-based prompt"),
    ("Domani il mondo", "A future-based prompt"),
    ("Lei mi ha detto", "A dialogue-based prompt"),
]

for prompt, description in test_prompts:
    generated = generate_text(model, tokenizer, prompt, max_length=25, temperature=0.9)
    print(f"\n  [{description}]")
    print(f"  Prompt:    '{prompt}'")
    print(f"  Generated: {generated}")
    print(f"  {'~' * 58}")

print(f"""
  ANALYSIS QUESTIONS for each generated line:
  
  1. Are the words real Italian words, or made-up?
  2. Does the grammar make sense (gender agreement, verb forms)?
  3. Is there a coherent meaning, or just a sequence of
     vaguely related words?
  4. Could you tell this was AI-generated if you didn't know?
  
  This is hallucination at a small scale. At the scale of
  ChatGPT, the same mechanism produces confident-sounding
  paragraphs about topics it has no real knowledge of.
""")
print("=" * 65)

### Why hallucination is so dangerous

The key problem with hallucination isn't that AI makes mistakes -- humans make mistakes too. The problem is that **AI makes mistakes with the same confidence as when it's correct.**

There's no hesitation, no "I'm not sure about this," no visible uncertainty. A hallucinated fact looks identical to a real one in the output. The model has no internal signal for "I'm making this up" versus "I know this is true" -- because it doesn't "know" anything. It just predicts likely next words.

This is why critical thinking remains essential when using AI tools. The output may look authoritative, but it always needs human verification for anything that matters.

---

## 5. Limitation 4: Environmental Cost

Training AI models consumes significant energy. This is a real and measurable cost that's worth understanding.

### Our project vs. the real world

**Our experiment** used your laptop or a small computer for a few hours. The electricity consumed was roughly equivalent to keeping a light bulb on for an afternoon. Totally negligible.

**Training GPT-4** reportedly required:
- Tens of thousands of specialized GPU chips running simultaneously
- Several months of continuous computation
- An estimated energy consumption equivalent to **hundreds of households for an entire year**
- An estimated cost of over **$100 million** in compute alone

And that's just the training. Every time someone asks ChatGPT a question, additional energy is consumed for inference (generating the response). With hundreds of millions of users, this adds up quickly.

### Putting it in perspective

| Model | Training Energy | Approximate CO2 | Equivalent To |
|-------|----------------|-----------------|---------------|
| Our LSTM | ~0.1 kWh | Negligible | Charging a phone |
| BERT (2018) | ~1,500 kWh | ~650 kg CO2 | One transatlantic flight |
| GPT-3 (2020) | ~1,300,000 kWh | ~550 tons CO2 | 120 cars for a year |
| GPT-4 (2023) | Not fully disclosed | Significantly more than GPT-3 | Estimated: hundreds of households |

*Note: These are estimates from published research and reporting. Exact figures for proprietary models are not always publicly available.*

### The nuance

It's important to note several things:

1. **Efficiency is improving rapidly.** Newer chips and training techniques are significantly more energy-efficient than older ones. The same capability can often be achieved with less energy each year.

2. **Scale is growing even faster.** But models are also getting bigger, and more people are using them. The efficiency gains are often outpaced by the growth in scale.

3. **The comparison matters.** If an AI system replaces a process that was itself energy-intensive (like millions of individual Google searches, or physical mail), the net environmental impact might actually be positive.

4. **Renewable energy helps.** Some AI companies run their data centers on renewable energy, which changes the environmental equation significantly.

The honest answer is: AI has a meaningful environmental footprint, it's growing, and it deserves attention -- but it's not an existential environmental crisis on its own. It's one factor among many in the broader conversation about technology and sustainability.

---

## 6. Limitation 5: The Data Problem

This limitation isn't technical -- it's ethical. And it might be the most important one.

### Who created the data?

Every AI model is trained on data created by humans. In our case, the data is Italian song lyrics -- written by real songwriters, performed by real artists, expressing real emotions from real lives.

Ask yourself:
- **Who wrote these songs?** They were professional and amateur musicians who spent hours, days, sometimes years crafting their lyrics.
- **Were they compensated** for their work being used to train an AI? Almost certainly not.
- **Did they consent** to their lyrics being included in a machine learning dataset? In most cases, probably not.
- **Would they approve** of an AI model learning to imitate their style? Some might be flattered; others might feel robbed.

### The broader copyright debate

These same questions are at the heart of a massive ongoing legal and ethical debate:

- **Visual artists** have filed lawsuits against AI image generators (like Stable Diffusion and Midjourney) for training on their artwork without permission
- **Authors and journalists** have sued AI companies for using their books and articles as training data
- **Musicians** are grappling with AI-generated songs that mimic specific artists' voices and styles
- **Programmers** have raised concerns about AI coding assistants trained on open-source code

There are no easy answers. The arguments on both sides have merit:

**In favor of using data for AI training:**
- Much of the data is publicly available on the internet
- AI training could be considered "fair use" (learning from, not copying)
- Restricting training data could slow beneficial AI development
- It's similar to how humans learn by reading and listening to others' work

**Against using data without consent:**
- Creators deserve to control how their work is used
- AI-generated content can directly compete with and replace the original creators
- "Publicly available" doesn't mean "free to use for any purpose"
- The economic value flows to AI companies, not to the creators whose work made it possible

### Why this matters for you

As someone who now understands how AI works, you're in a better position than most people to think critically about these questions. When you use an AI tool, you can ask:

- What data was this trained on?
- Were the creators of that data compensated or consulted?
- Am I using AI in a way that respects the people whose work made it possible?

There are no simple answers, but asking the questions is where responsible use begins.

---

## 7. Opportunities: What AI Makes Possible

We've spent a lot of this notebook discussing what can go wrong. That's important -- but it's only half the picture. AI also creates genuine, transformative opportunities. Let's look at them honestly.

### Creative Tools: AI as a Collaborator

AI doesn't have to replace human creativity -- it can enhance it.

- **Songwriters** can use AI to brainstorm lyrics, explore rhyme schemes, or break through writer's block. The AI suggests; the human decides.
- **Visual artists** can use AI to rapidly prototype ideas, generate variations, or explore styles they wouldn't have tried otherwise.
- **Writers** can use AI to draft, edit, restructure, or find the right word when it's on the tip of their tongue.

The key insight: AI is most powerful not as a replacement for human creativity, but as a **tool that amplifies it** -- like how a calculator doesn't replace mathematical thinking, but frees mathematicians to focus on harder problems.

### Language Access: Breaking Down Barriers

Language AI has the potential to make the world more accessible:

- **Translation** between languages is becoming remarkably good, enabling communication across cultures
- **Language learning** tools can provide personalized practice conversations in any language, at any hour
- **Accessibility** tools can convert speech to text, text to speech, describe images for blind users, and simplify complex documents for people with cognitive disabilities
- **Preservation** of endangered languages by training models on small corpora of rare languages

### Education: Personalized Learning

AI can adapt to individual learners in ways that a single teacher with 30 students cannot:

- **Personalized tutoring** that adjusts to a student's pace, identifies weak spots, and provides targeted practice
- **Making complex topics accessible** -- like this very project, which uses Italian song lyrics to teach the fundamentals of AI
- **Creating educational content** in multiple languages and at different reading levels
- **Answering questions patiently**, as many times as needed, without judgment

### Scientific Discovery: Accelerating Research

Some of AI's most impressive contributions are in scientific research:

- **Drug discovery**: AI models can predict which molecular structures might be effective medicines, dramatically reducing the time and cost of bringing new treatments to patients
- **Climate modeling**: AI helps process the enormous datasets needed to understand and predict climate change
- **Protein folding**: DeepMind's AlphaFold solved a 50-year-old biology problem by predicting how proteins fold into 3D shapes -- a breakthrough with implications for medicine, agriculture, and energy
- **Materials science**: AI is helping discover new materials for batteries, solar panels, and other technologies

### Productivity: Focusing on What Matters

AI can handle repetitive tasks so humans can focus on creative and meaningful work:

- **Summarizing** long documents, emails, and meeting notes
- **Automating** routine data entry, scheduling, and administrative tasks
- **Coding assistance** that helps programmers write, debug, and understand code faster
- **Customer service** that handles common questions instantly, freeing human agents for complex situations

### The common thread

Notice something about all these opportunities: **they work best when AI and humans work together.** The AI handles the parts it's good at (processing large amounts of data, generating options quickly, working tirelessly), and the human handles the parts that require judgment, creativity, ethics, and genuine understanding.

This isn't a future prediction -- it's happening right now, in every one of these domains.

---

## 8. The Scale Perspective

One final comparison to put everything in context. Throughout these notebooks, you've built models that use the exact same fundamental mechanism as the most powerful AI systems in the world. The difference? Scale.

Let's see just how dramatic that difference is.

In [ ]:
# ================================================================
#  THE SCALE PERSPECTIVE: Our project vs. the real world
# ================================================================

# Calculate our model's parameter count
our_model = LSTMModel(vocab_size=50257, embed_dim=256, hidden_dim=512)
our_params = sum(p.numel() for p in our_model.parameters())

print("=" * 70)
print("  FROM OUR LAPTOP TO GPT-4: The Power of Scale")
print("=" * 70)

# Format large numbers for readability
def format_number(n):
    if n >= 1_000_000_000_000:
        return f"{n/1_000_000_000_000:.1f} trillion"
    elif n >= 1_000_000_000:
        return f"{n/1_000_000_000:.1f} billion"
    elif n >= 1_000_000:
        return f"{n/1_000_000:.1f} million"
    elif n >= 1_000:
        return f"{n/1_000:.1f} thousand"
    return str(n)

# The comparison data
models = [
    {
        "name": "Our LSTM",
        "params": our_params,
        "training_data": "~9,135 songs",
        "training_time": "~30 minutes",
        "cost": "$0 (your laptop)",
        "capability": "Italian-ish word sequences"
    },
    {
        "name": "Our GPT-2 (fine-tuned)",
        "params": 124_000_000,
        "training_data": "~9,135 songs",
        "training_time": "~2 hours",
        "cost": "$0 (your laptop)",
        "capability": "Better Italian lyrics"
    },
    {
        "name": "GPT-2 (original)",
        "params": 1_500_000_000,
        "training_data": "~8 million web pages",
        "training_time": "~1 week",
        "cost": "~$50,000",
        "capability": "Coherent paragraphs"
    },
    {
        "name": "GPT-3",
        "params": 175_000_000_000,
        "training_data": "~500 billion tokens",
        "training_time": "~1 month",
        "cost": "~$4.6 million",
        "capability": "Impressive text generation"
    },
    {
        "name": "GPT-4 (estimated)",
        "params": 1_800_000_000_000,
        "training_data": "Trillions of tokens",
        "training_time": "~3-6 months",
        "cost": "~$100 million",
        "capability": "Near-human text, reasoning"
    },
]

for m in models:
    print(f"\n  {'~' * 64}")
    print(f"  {m['name']}")
    print(f"  {'~' * 64}")
    print(f"    Parameters:    {format_number(m['params'])}")
    print(f"    Training data: {m['training_data']}")
    print(f"    Training time: {m['training_time']}")
    print(f"    Cost:          {m['cost']}")
    print(f"    Capability:    {m['capability']}")

# The scale comparison
print(f"\n{'=' * 70}")
print(f"  THE SCALE DIFFERENCE")
print(f"{'=' * 70}")

ratio = 1_800_000_000_000 / our_params
print(f"""
  Our LSTM has {format_number(our_params)} parameters.
  GPT-4 has an estimated {format_number(1_800_000_000_000)} parameters.
  
  That's roughly {ratio:,.0f}x more parameters.
  
  Same fundamental mechanism: predict the next word.
  Radically different capabilities.
  
  Scale matters. A lot.
""")
print("=" * 70)

### What scale teaches us

The comparison above is striking. Our tiny model and GPT-4 use the same fundamental idea -- predict the next token -- but the difference in capability is enormous.

This raises a deep question that researchers are still debating: **Is intelligence just a matter of scale?** If you make a next-word predictor big enough and feed it enough data, does it eventually become "intelligent"? Or is something fundamentally different needed?

We don't know the answer yet. But what you've seen in these notebooks gives you the foundation to think about this question with real understanding, not just opinions based on headlines.

---

## 9. What You've Learned: The Complete Journey

Let's step back and appreciate how far you've come. Over seven notebooks, you've gone from zero to a genuine understanding of how modern AI works. Here's the complete journey:

### Notebook 01: Exploring the Data
**AI starts with data -- the quality and composition of data shapes everything.**
You loaded a real dataset of Italian song lyrics, explored its structure, and saw firsthand that data isn't neutral. What's included and what's excluded determines what the AI can and cannot do.

### Notebook 02: How AI Reads Text
**Text must become numbers (tokenization) before AI can process it.**
You learned that computers don't understand words -- they understand numbers. You saw how tokenization breaks text into pieces and converts them into numerical representations that a neural network can work with.

### Notebook 03: Training a Simple RNN
**AI learns by predicting the next word and adjusting when wrong.**
You built your first neural network -- a Recurrent Neural Network -- and watched it learn from scratch. You saw the training loop: predict, compare to reality, calculate the error, adjust the weights, repeat millions of times.

### Notebook 04: LSTM -- Better Memory
**Better memory architectures improve coherence.**
You discovered that simple RNNs forget things quickly, and you built an LSTM that can remember information over longer sequences. The result: more coherent, more natural-sounding generated text.

### Notebook 05: The Transformer Revolution
**Transformers see all words at once -- the architecture behind modern AI.**
You explored the Transformer architecture and its key innovation: attention. Instead of reading words one at a time, Transformers can look at all words simultaneously and learn which ones are most relevant to each other. This is the architecture behind GPT, BERT, and every major language model.

### Notebook 06: Improving Generation
**Smart sampling strategies prevent repetition and improve quality.**
You learned that how you select words from the model's predictions matters enormously. Temperature, top-k, and top-p sampling transform repetitive, boring output into diverse, interesting text -- without changing the model at all.

### Notebook 07: Limitations, Ethics, and Opportunities (this notebook)
**AI has real limitations, biases, and costs -- but also enormous potential.**
You've seen that AI doesn't truly understand, that it reflects the biases in its training data, that it hallucinates with confidence, that it has environmental costs, and that it raises unresolved ethical questions about data ownership. But you've also seen the genuine opportunities it creates across creativity, education, science, and accessibility.

---

**That's a remarkable amount of knowledge.** You now understand the core ideas behind the AI systems that are reshaping the world. Not at a surface level -- at a fundamental, "I've seen how this actually works" level.

---

## 10. Where to Go From Here

Your journey doesn't have to end here. Here are some concrete suggestions for continuing to learn:

### Experiment with the code
- **Change the dataset**: What happens if you train on English lyrics? On poetry? On news articles? The code works with any text -- try it and see how the output changes.
- **Adjust the parameters**: Make the model bigger or smaller. Train for more or fewer epochs. Change the temperature. Each change teaches you something.
- **Try different prompts**: What kinds of prompts produce the best output? The worst? Why?

### Read the foundational papers
- **"Attention Is All You Need"** (Vaswani et al., 2017) -- The original Transformer paper. Now that you understand attention from Notebook 5, you'll be able to follow the core ideas. Don't worry about understanding every equation; focus on the architecture diagrams and the intuition.
- **"Language Models are Few-Shot Learners"** (Brown et al., 2020) -- The GPT-3 paper. It shows how scaling up the approach you've learned leads to remarkable emergent capabilities.

### Explore Hugging Face
- Visit [huggingface.co](https://huggingface.co) -- it's the largest repository of pre-trained AI models in the world. You can find thousands of models for text generation, translation, sentiment analysis, image generation, and more.
- Many models can be tried directly in your browser, no code required.
- The documentation includes excellent tutorials for beginners.

### Think critically about AI in the news
- When you read an article about AI, ask yourself: What model architecture are they using? What data was it trained on? What are the limitations they're not mentioning?
- You now have the vocabulary and understanding to evaluate AI claims. Use it.
- Be skeptical of both extreme hype ("AI will solve everything") and extreme fear ("AI will destroy everything"). The reality is more nuanced, as you've seen.

### Consider the ethical implications in your own field
- Whatever your profession or area of interest, AI is probably affecting it or will soon. How?
- What data in your field could be used to train AI? Who would benefit? Who might be harmed?
- What tasks in your work could AI assist with? What tasks should remain human?

### Keep learning
- **Fast.ai** (fast.ai) offers free, practical courses on deep learning that build on everything you've learned here
- **3Blue1Brown** (YouTube) has excellent visual explanations of neural networks and linear algebra
- **Stanford CS224N** (free online) is the gold-standard university course on Natural Language Processing

---

## 11. Final Thought

> **The most important thing to remember: AI is a tool built by humans, trained on human data, with human choices at every step. Understanding how it works -- as you now do -- is the first step to using it wisely, shaping it responsibly, and ensuring it benefits everyone.**

You started this series by looking at a dataset of Italian songs. You ended it by understanding the technology that is reshaping how we work, create, learn, and communicate.

Along the way, you saw that there's no magic behind AI -- just mathematics, data, and clever engineering. The same next-word prediction that powers our little Italian lyrics generator is the mechanism behind the most powerful AI systems ever built. The difference is scale, not principle.

You also saw that this technology comes with real limitations and real responsibilities. It doesn't understand. It reflects our biases. It hallucinates. It costs energy. It raises questions about fairness and ownership that we haven't yet answered.

But you also saw the potential: for creativity, for accessibility, for scientific discovery, for education.

The future of AI will be shaped by people who understand it -- not just the engineers who build it, but everyone who uses it, regulates it, and lives with its consequences.

**You are now one of those people.**

---

*Thank you for following along. Grazie mille.*

---

*End of Notebook 7 of 7. End of the series.*